In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix, precision_score, recall_score 
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC


In [2]:
df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")



In [3]:
'''df = df.drop(['id', 'Soil_pH', 'Organic_Carbon', 'Electrical_Conductivity', 'Sunlight_Hours', 'Region_West', 'Region_South', 'Region_North', 'Region_East', 'Region_Central'], errors='ignore')
print(df)'''

"df = df.drop(['id', 'Soil_pH', 'Organic_Carbon', 'Electrical_Conductivity', 'Sunlight_Hours', 'Region_West', 'Region_South', 'Region_North', 'Region_East', 'Region_Central'], errors='ignore')\nprint(df)"

In [4]:
#train-test split
x = df.drop(['Irrigation_Need'], axis =1)
y = df['Irrigation_Need']
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.2, random_state = 42
)

In [5]:
numeric_columns = df.select_dtypes(include= ['int64', 'float64']).columns

categorical_columns = df.select_dtypes(include = ['object']).columns
categorical_columns = categorical_columns.drop('Irrigation_Need', errors='ignore')


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers = [
        ('onehot',OneHotEncoder(sparse_output = False),categorical_columns),
        
    ],
    remainder = 'passthrough'
)

 
pipeline = Pipeline([('encoding', preprocessor),('model', StackingClassifier(estimators= [('rf', RandomForestClassifier(n_estimators =100, random_state=42)),
                                                                                         ('svc', SVC(C = 1, kernel = 'linear',probability = True))],
                   final_estimator = LogisticRegression()
                   ))
])

pipeline.fit(x_train, y_train)
y_pred = pipeline.predict(x_test)

In [ ]:
print("BalancedAccuracy:", balanced_accuracy_score(y_test, y_pred) * 100)
print("Confusion Matrix:\n",
confusion_matrix(y_test, y_pred))
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print("Precision:", precision)
print("Recall:", recall)

In [ ]:
'''#decision tree
from sklearn.tree import DecisionTreeClassifier
model1 = DecisionTreeClassifier(random_state = 42)
model1.fit(x_train,y_train)
y_pred = model1.predict(x_test)'''

In [ ]:
test_df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/test.csv")
y_pred = pipeline.predict(x_test)
print(y_pred)


In [ ]:
submission = pd.Dataframe({
    'id' = test_df['id'],
    'Irrigation_Need': y_pred

})